# 🪝 Module 1.5 — Agent SDK Hooks

**Domain 1 · Agentic Architecture & Orchestration** (27% of the exam)
**Task 1.5 · Agent SDK Hooks**
**Source:** [claudecertificationguide.com/learn/1-agentic-architecture/1-5-agent-sdk-hooks](https://claudecertificationguide.com/learn/1-agentic-architecture/1-5-agent-sdk-hooks)

Module 1.4 built enforcement by hand — a plain Python `if` statement wrapped
around a tool function. This module is the real, SDK-native version of that
same idea: **hooks** that intercept a tool call before or after it runs,
without touching the tool's own code at all.

### 🎯 What you'll build

Six in-process tools (three with deliberately incompatible data formats,
three for a refund/transfer/compliance scenario), a `PostToolUse` hook that
normalizes the messy ones, and two `PreToolUse` hooks that enforce a refund
threshold and an AML compliance gate — all running for real against
Claude Code's actual hook dispatch, not a simulation of it.

### ✅ What you'll walk away knowing

1. `PostToolUse` transforms data *after* a tool runs; `PreToolUse` enforces
   policy *before* one runs — and why that direction is never optional
2. Why `PostToolUse` can log a violation but can never undo one
3. The same 100%-vs-probabilistic decision rule from Module 1.4, now applied
   at the hook layer
4. How to normalize heterogeneous tool outputs so the model stops having to
   guess at formats
5. How a `PreToolUse` gate and a `PostToolUse` state-setter work together to
   enforce a multi-step compliance requirement (AML before transfer)

---

> **⚠️ The most experimental notebook in this repo so far — read this before
> running it.** Every type and field below is verified against the actually
> installed `claude-agent-sdk` package, including two of Anthropic's own
> docstring examples that *corrected* an initial assumption of mine (more on
> that in Task 1). What I could **not** do is run this end-to-end against a
> live API during generation — that would spend your API budget without
> asking, which this project's own conventions rule out. So the hooks below
> are written defensively on purpose (matching tool names by suffix rather
> than assuming one exact naming convention, parsing tool responses
> tolerantly rather than assuming one exact shape) — if something about the
> exact runtime wiring is slightly off on your first real run, those are the
> two places to look first, and the verbose print statements throughout are
> there specifically to make that debugging fast.
>
> **💳 Cost:** 3 real `query()` calls doing real tool orchestration — no web
> search this time, so expect Module 1.1/1.2 territory (cheap, fast), not
> Module 1.3's.

## 🔧 Setup

Same package as Module 1.3.

```bash
pip install claude-agent-sdk
```

```bash
# Windows (PowerShell)
$env:ANTHROPIC_API_KEY = "sk-ant-..."

# macOS / Linux (bash/zsh)
export ANTHROPIC_API_KEY="sk-ant-..."
```


In [ ]:
import os
import json

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Set it in your shell, then restart the "
        "kernel and run this cell again -- see the Setup section above."
    )

from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    HookMatcher,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
)

print("claude_agent_sdk imported. Ready.")


## 🔑 Key Concept: Hook Direction Is the Whole Question

| | `PreToolUse` | `PostToolUse` |
|---|---|---|
| **Fires** | Before the tool executes | After the tool executes, before the model sees the result |
| **Can it block?** | Yes — full control over whether the call happens at all | No — the side effect has already happened |
| **What it returns** | `permissionDecision`: `allow` \| `deny` \| `ask` \| `defer`, plus optional `updatedInput` | `updatedToolOutput` (replaces what the model sees) |
| **Used for** | Policy enforcement | Data transformation / normalization |

> "PostToolUse hooks transform data **after** execution. PreToolUse hooks
> enforce policy **before** execution. Know which direction each hook
> operates in — the exam tests this distinction."

The consequence that matters most: **by the time `PostToolUse` fires, the
tool has already run.** Blocking there stops the *loop*, not the *action* —
the refund already went out, the transfer already happened. If you need to
prevent something, it has to be `PreToolUse`.


## 🔑 Key Concept: The Same Decision Rule, One Layer Down

| Requirement | Mechanism | Guarantee |
|---|---|---|
| Must be followed 100% of the time | Hook | Deterministic |
| Preferred, occasional deviation is fine | Prompt | Probabilistic |

> "If the business would lose money from a single failure → use a hook. If
> the business would face legal risk from a single failure → use a hook. If
> it is a formatting preference or style guideline → prompt-based guidance
> is fine."

This is Module 1.4's enforcement spectrum again — hooks are just *where*
that deterministic enforcement actually lives when you're building on the
Agent SDK instead of hand-rolling a loop.


## 🔑 Key Concept: `PostToolUse` for Data Normalization

Different tools return the same *kind* of information in incompatible
shapes: Unix epoch vs. ISO 8601 timestamps, numeric vs. string vs.
single-character status codes, `DD/MM/YYYY` vs. other date orders. Left
alone, "the model must interpret mixed formats repeatedly, causing
inconsistency. It might parse a Unix timestamp correctly one time and
misread it the next."

**The fix:** a `PostToolUse` hook intercepts every relevant tool's raw
result and rewrites it into one consistent schema — before the model ever
sees the inconsistency — via `updatedToolOutput`.

### Case study we'll build against: the data-format-chaos scenario

Three tools, three formats:

| Tool | Date format | Status format |
|---|---|---|
| `get_customer` | Unix epoch (`1710489600`) | Numeric (`0`, `1`, `2`) |
| `lookup_order` | ISO 8601 (`"2024-03-15T12:00:00Z"`) | English string (`"shipped"`) |
| `check_shipping` | `DD/MM/YYYY` (`"15/03/2024"`) | Single character (`"S"`, `"P"`, `"D"`) |

Common real failures without normalization: confusing day/month order,
reading `"P"` as "processed" instead of "pending", or converting Unix
timestamps inconsistently across iterations.


## 🔑 Key Concept: `PreToolUse` for Policy Enforcement

Three canonical uses:

1. **Threshold enforcement** — deny `process_refund` above $500, route to a
   human instead.
2. **Compliance prerequisite gates** — deny `transfer_funds` until
   `aml_check` has passed *in this session* — works "100% of the time. No
   transfer can execute without AML verification."
3. **Approval workflows** — `permissionDecision: "ask"` pauses a
   >20% `approve_discount` for a human decision instead of outright denying it.

All three share the same shape: intercept, inspect the tool's own input,
decide, and — critically — the tool's real handler never even runs for a
denied call.


## 🔑 Key Concept: Hooks vs. Prompts, Side by Side

| Scenario | Prompt approach | Hook approach | Verdict |
|---|---|---|---|
| International transfers need AML checks | ~95% success; failures are a regulatory violation | `PreToolUse` blocks until `aml_check` passes: 100% | **Hook required** |
| Format all responses in markdown | High success; occasional plain text isn't a business risk | `PostToolUse` reformatting: 100% but unnecessary complexity | **Prompt is fine** |
| Refunds over $500 need human escalation | Works most of the time; one failure = an unapproved large refund | `PreToolUse` blocks and routes to escalation: 100% | **Hook required** |

The pattern: **cost of a single failure** decides this, every time — not
how often the failure happens to occur.


## 🛠️ Build Exercise — Task 1: Six Tools, Three of Them Deliberately Messy

**Objective:** `get_customer`, `lookup_order`, `check_shipping` (the
format-chaos trio) plus `process_refund`, `transfer_funds`, `aml_check` (the
enforcement trio) — all as real, in-process tools via the SDK's `@tool`
decorator.

**Why this matters:** this recreates the exact scenario the exam builds its
questions around — you can't appreciate why normalization and gating matter
until you've felt three inconsistent formats collide in one conversation.

> **A real correction, caught by reading the SDK's own docstrings rather
> than assuming:** `create_sdk_mcp_server`'s own example registers tools in
> `allowed_tools` by their **bare** name (`allowed_tools=["add", "multiply"]`)
> — not a `mcp__servername__toolname`-prefixed form I initially expected
> from other MCP contexts. We follow that verified, documented example
> directly below.


In [ ]:
# --- The format-chaos trio -------------------------------------------------

@tool("get_customer", "Look up a customer by ID.", {"customer_id": str})
async def get_customer(args):
    customer_id = args["customer_id"]
    # Unix epoch timestamp + numeric status code (0=inactive, 1=active, 2=pending)
    data = {"customer_id": customer_id, "last_contact_unix": 1710489600, "status_code": 1}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


@tool("lookup_order", "Look up an order by ID.", {"order_id": str})
async def lookup_order(args):
    order_id = args["order_id"]
    # ISO 8601 date + English status string already -- this one's already clean.
    data = {"order_id": order_id, "ship_date_iso": "2024-03-15T12:00:00Z", "status": "shipped"}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


@tool("check_shipping", "Check shipping status by tracking ID.", {"tracking_id": str})
async def check_shipping(args):
    tracking_id = args["tracking_id"]
    # DD/MM/YYYY date + single-character status (S=shipped, P=pending, D=delivered)
    data = {"tracking_id": tracking_id, "date_dmy": "15/03/2024", "status_code": "S"}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


# --- The enforcement trio ---------------------------------------------------

@tool("process_refund", "Process a refund for a customer.", {"customer_id": str, "amount": float})
async def process_refund(args):
    data = {"status": "refunded", "customer_id": args["customer_id"], "amount": args["amount"]}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


@tool("transfer_funds", "Transfer funds from an account.", {"account_id": str, "amount": float})
async def transfer_funds(args):
    data = {"status": "transferred", "account_id": args["account_id"], "amount": args["amount"]}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


@tool("aml_check", "Run an AML (anti-money-laundering) check on an account.", {"account_id": str})
async def aml_check(args):
    data = {"account_id": args["account_id"], "aml_status": "pass"}
    return {"content": [{"type": "text", "text": json.dumps(data)}]}


support_tools_server = create_sdk_mcp_server(
    name="support_tools",
    tools=[get_customer, lookup_order, check_shipping, process_refund, transfer_funds, aml_check],
)

ALL_TOOL_NAMES = ["get_customer", "lookup_order", "check_shipping", "process_refund", "transfer_funds", "aml_check"]
print("Registered SDK MCP server with tools:", ALL_TOOL_NAMES)


## 🛠️ Build Exercise — Task 2: `PostToolUse` Normalization Hook

**Objective:** one hook that intercepts all three format-chaos tools and
rewrites their output to a single schema: ISO 8601 dates, English status
strings.

**Why this matters:** this is the exam's tested direction for
transformation — after execution, before the model reads the result.

Two defensive choices worth calling out, both there because of the
uncertainty flagged at the top of this notebook:
- **Matching by suffix**, not an assumed exact tool-name string — robust
  whether the runtime exposes tools as `"get_customer"` or under some
  prefixed form.
- **Tolerant response parsing** — a tool's `tool_response` is unwrapped
  defensively rather than assuming one fixed shape.


In [ ]:
def _extract_tool_json(tool_response) -> dict:
    """Tool responses are wrapped per the MCP protocol; unwrap defensively
    since the exact shape reaching a hook isn't something we've been able
    to confirm against a live run (see the notebook's opening note).
    """
    if isinstance(tool_response, dict) and "content" in tool_response:
        for block in tool_response["content"]:
            if isinstance(block, dict) and block.get("type") == "text":
                try:
                    return json.loads(block["text"])
                except (ValueError, TypeError):
                    pass
    if isinstance(tool_response, str):
        try:
            return json.loads(tool_response)
        except ValueError:
            pass
    if isinstance(tool_response, dict):
        return tool_response
    return {}


_STATUS_NUMERIC_MAP = {0: "inactive", 1: "active", 2: "pending"}
_STATUS_CHAR_MAP = {"S": "shipped", "P": "pending", "D": "delivered"}


def _normalize_date(record: dict) -> str | None:
    if "last_contact_unix" in record:
        import datetime
        return datetime.datetime.fromtimestamp(
            record["last_contact_unix"], tz=datetime.timezone.utc
        ).strftime("%Y-%m-%dT%H:%M:%SZ")
    if "ship_date_iso" in record:
        return record["ship_date_iso"]
    if "date_dmy" in record:
        day, month, year = record["date_dmy"].split("/")
        return f"{year}-{month}-{day}T00:00:00Z"
    return None


def _normalize_status(record: dict) -> str | None:
    if "status_code" in record and isinstance(record["status_code"], int):
        return _STATUS_NUMERIC_MAP.get(record["status_code"], "unknown")
    if "status_code" in record and isinstance(record["status_code"], str):
        return _STATUS_CHAR_MAP.get(record["status_code"], "unknown")
    if "status" in record:
        return record["status"]
    return None


async def normalize_data_hook(input_data, tool_use_id, context):
    """PostToolUse -- runs AFTER the tool executes. Cannot undo anything;
    can only change what the model sees next.
    """
    tool_name = input_data.get("tool_name", "")
    if not any(tool_name.endswith(name) for name in ("get_customer", "lookup_order", "check_shipping")):
        return {}  # not one of the format-chaos tools; no opinion

    record = _extract_tool_json(input_data.get("tool_response"))
    normalized = dict(record)
    date_value = _normalize_date(record)
    status_value = _normalize_status(record)
    if date_value is not None:
        normalized["date"] = date_value
    if status_value is not None:
        normalized["status"] = status_value

    print(f"  [PostToolUse] normalized {tool_name}: {record} -> {normalized}")

    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "updatedToolOutput": {"content": [{"type": "text", "text": json.dumps(normalized)}]},
        }
    }


## 🛠️ Build Exercise — Task 3: Verify Consistent Data, For Real

**Objective:** run a real query that touches all three format-chaos tools,
and confirm the model is working from normalized data throughout.

This makes one real API call.


In [ ]:
normalization_options = ClaudeAgentOptions(
    mcp_servers={"support_tools": support_tools_server},
    allowed_tools=["get_customer", "lookup_order", "check_shipping"],
    hooks={"PostToolUse": [HookMatcher(matcher=None, hooks=[normalize_data_hook])]},
)

NORMALIZATION_PROMPT = (
    "Look up customer CUST-1, order ORD-1, and shipping tracking TRACK-1 "
    "(using get_customer, lookup_order, and check_shipping respectively), "
    "then summarize the date and status you got back from each one."
)

final_summary = None

async for message in query(prompt=NORMALIZATION_PROMPT, options=normalization_options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[assistant] {block.text}")
            elif isinstance(block, ToolUseBlock):
                print(f"[assistant -> tool_use] {block.name}({block.input})")
    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print(f"[tool_result for {block.tool_use_id}] {str(block.content)[:200]}")
    elif isinstance(message, ResultMessage):
        final_summary = message.result
        print()
        print("=== Final summary ===")
        print(message.result)
        if message.total_cost_usd is not None:
            print(f"Cost: ${message.total_cost_usd:.4f}")

print()
print("Look at the [PostToolUse] lines printed above (from inside the hook itself) --")
print("each one shows the RAW tool output next to the NORMALIZED version. If the final")
print("summary references dates/statuses consistently despite three different source")
print("formats, normalization worked end to end.")


## 🛠️ Build Exercise — Task 4: `PreToolUse` Refund Threshold

**Objective:** deny `process_refund` above $500, before it ever executes.

**Why this matters:** this is the exam's central warning, restated at the
hook layer — `PostToolUse` would be too late here; the refund would already
be gone.


In [ ]:
REFUND_THRESHOLD = 500.0


async def refund_threshold_hook(input_data, tool_use_id, context):
    """PreToolUse -- runs BEFORE the tool executes. A deny here means
    process_refund's real handler never runs at all for this call.
    """
    tool_name = input_data.get("tool_name", "")
    if not tool_name.endswith("process_refund"):
        return {}

    amount = input_data.get("tool_input", {}).get("amount", 0)
    if amount > REFUND_THRESHOLD:
        print(f"  [PreToolUse] DENYING refund of ${amount} (exceeds ${REFUND_THRESHOLD} threshold)")
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (
                    f"Refund of ${amount} exceeds the ${REFUND_THRESHOLD} threshold -- "
                    f"escalate to a human agent for approval."
                ),
            }
        }

    print(f"  [PreToolUse] allowing refund of ${amount} (within threshold)")
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "allow"}}


## 🛠️ Build Exercise — Task 5: `PreToolUse` + `PostToolUse` AML Gate

**Objective:** `transfer_funds` is denied until `aml_check` has passed *in
this session* — a `PreToolUse` gate paired with a `PostToolUse` hook that
records the pass.

**Why this matters:** this is the module's central compliance scenario.
Note the two hooks working together, exactly like Module 1.4's Python-level
gate — except this time both halves are real SDK hooks instead of a
hand-wrapped function.


In [ ]:
# Plain Python state -- lives in THIS process, visible to hook callbacks via
# closure, same idea as Module 1.4's session_state dict.
aml_passed_accounts = set()


async def aml_gate_hook(input_data, tool_use_id, context):
    """PreToolUse on transfer_funds."""
    tool_name = input_data.get("tool_name", "")
    if not tool_name.endswith("transfer_funds"):
        return {}

    account_id = input_data.get("tool_input", {}).get("account_id")
    if account_id not in aml_passed_accounts:
        print(f"  [PreToolUse] DENYING transfer for {account_id} -- no AML pass on record")
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (
                    f"No AML check on record for account {account_id}. "
                    f"Run aml_check first, then retry the transfer."
                ),
            }
        }

    print(f"  [PreToolUse] allowing transfer for {account_id} -- AML already passed")
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse", "permissionDecision": "allow"}}


async def aml_record_pass_hook(input_data, tool_use_id, context):
    """PostToolUse on aml_check -- records a pass so the gate above can see it."""
    tool_name = input_data.get("tool_name", "")
    if not tool_name.endswith("aml_check"):
        return {}

    result = _extract_tool_json(input_data.get("tool_response"))
    if result.get("aml_status") == "pass":
        aml_passed_accounts.add(result.get("account_id"))
        print(f"  [PostToolUse] recorded AML pass for {result.get('account_id')}")
    return {}


## 🛠️ Build Exercise — Task 6: Test Both Hooks With Blocked Scenarios

**Objective:** confirm denied operations never execute, and the same
operation succeeds once its prerequisite is met.

Two real calls: one exercising the refund threshold, one exercising the AML
gate end to end (denied → run `aml_check` → retry → allowed).


In [ ]:
enforcement_options = ClaudeAgentOptions(
    mcp_servers={"support_tools": support_tools_server},
    allowed_tools=["process_refund", "transfer_funds", "aml_check"],
    hooks={
        "PreToolUse": [HookMatcher(matcher=None, hooks=[refund_threshold_hook, aml_gate_hook])],
        "PostToolUse": [HookMatcher(matcher=None, hooks=[aml_record_pass_hook])],
    },
)

REFUND_TEST_PROMPT = (
    "Process a $750 refund for customer CUST-2 using process_refund. If that "
    "doesn't go through, try a $100 refund for the same customer instead."
)

async for message in query(prompt=REFUND_TEST_PROMPT, options=enforcement_options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[assistant] {block.text}")
            elif isinstance(block, ToolUseBlock):
                print(f"[assistant -> tool_use] {block.name}({block.input})")
    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print(f"[tool_result for {block.tool_use_id}] {str(block.content)[:200]}")
    elif isinstance(message, ResultMessage):
        print()
        print("=== Final ===")
        print(message.result)

print()
print("Look for a [PreToolUse] DENYING line above for the $750 attempt, and an")
print("allowing line for the $100 one -- that's the gate working before either")
print("call ever reached process_refund's real handler.")


In [ ]:
AML_TEST_PROMPT = (
    "Transfer $1000 from account ACC-9 using transfer_funds. If that's denied "
    "because of an AML check, run aml_check on ACC-9, then retry the transfer."
)

async for message in query(prompt=AML_TEST_PROMPT, options=enforcement_options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(f"[assistant] {block.text}")
            elif isinstance(block, ToolUseBlock):
                print(f"[assistant -> tool_use] {block.name}({block.input})")
    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock):
                print(f"[tool_result for {block.tool_use_id}] {str(block.content)[:200]}")
    elif isinstance(message, ResultMessage):
        print()
        print("=== Final ===")
        print(message.result)

print()
print(f"aml_passed_accounts now contains: {aml_passed_accounts}")
print("The first transfer_funds attempt should have been denied (no AML pass yet);")
print("after aml_check ran, the retry should have been allowed -- two real SDK hooks")
print("cooperating across separate tool calls, exactly like Module 1.4's gate did")
print("by hand.")


## ⚠️ Four Anti-Patterns to Avoid

| # | Anti-pattern | Why it fails | Fix |
|---|---|---|---|
| 1 | Using `PostToolUse` to block a policy violation | The tool already ran — there's no side effect left to prevent | `PreToolUse`, always, for anything that must not happen |
| 2 | A stronger, more detailed prompt for 100% compliance | Still probabilistic — ~95% is the ceiling, not 100% | A hook, same as Module 1.4's lesson |
| 3 | Relying on the model to normalize formats itself | Reintroduces the exact inconsistency a hook exists to remove | `PostToolUse` normalization (Task 2) |
| 4 | Mixing up which hook does which job | Either you miss the chance to prevent something, or you block work that's already done and can't be undone | `PreToolUse` = before/enforce, `PostToolUse` = after/transform |

Each is written below as real code, then commented out.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 1 -- PostToolUse "blocking" a policy violation
# ============================================================
# Commented out on purpose.
#
# async def refund_blocker_antipattern(input_data, tool_use_id, context):
#     # PostToolUse -- by the time this fires, process_refund has ALREADY RUN.
#     if input_data.get("tool_name", "").endswith("process_refund"):
#         amount = input_data.get("tool_input", {}).get("amount", 0)
#         if amount > REFUND_THRESHOLD:
#             return {"hookSpecificOutput": {"hookEventName": "PostToolUse",
#                                             "additionalContext": "This refund should have been blocked!"}}
#     return {}
#
# Why it fails: PostToolUse has no permissionDecision field at all -- there is
# nothing to "deny" after the fact. The refund already happened; at best this
# can flag it for someone to notice later, which is a very different (and
# much weaker) guarantee than actually preventing it.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 2 -- A stronger prompt instead of a hook
# ============================================================
# Commented out on purpose.
#
# STRONGER_SYSTEM_PROMPT = (
#     "CRITICAL COMPLIANCE REQUIREMENT: you MUST run aml_check and confirm a "
#     "pass BEFORE EVER calling transfer_funds. This is MANDATORY and "
#     "NON-NEGOTIABLE. Failure to comply is a serious violation."
# )
#
# Why it fails: same lesson as Module 1.4 -- capitalization and urgency
# improve a probability, they don't eliminate it. A hook physically prevents
# the call; a prompt, however forceful, only ever asks nicely.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 3 -- Relying on the model to normalize formats itself
# ============================================================
# Commented out on purpose.
#
# NORMALIZE_VIA_PROMPT = (
#     "When you get results from get_customer, lookup_order, or "
#     "check_shipping, please convert all dates to ISO 8601 and all statuses "
#     "to plain English before using them."
# )
#
# Why it fails: this is exactly the inconsistency Task 2's hook exists to
# remove -- "it might parse a Unix timestamp correctly one time and misread "
# "it the next." Asking the model to do the normalization is asking it to be
# the thing that's unreliable here, on every single tool call.


In [ ]:
# ============================================================
# ❌ ANTI-PATTERN 4 -- Registering the hooks under the wrong event
# ============================================================
# Commented out on purpose.
#
# wrong_direction_options = ClaudeAgentOptions(
#     mcp_servers={"support_tools": support_tools_server},
#     hooks={
#         "PreToolUse": [HookMatcher(matcher=None, hooks=[normalize_data_hook])],   # wrong event for this hook
#         "PostToolUse": [HookMatcher(matcher=None, hooks=[refund_threshold_hook])], # wrong event for this hook
#     },
# )
#
# Why it fails: normalize_data_hook reads tool_response, which only exists
# on PostToolUse -- registered under PreToolUse it would find nothing to
# normalize. refund_threshold_hook returns permissionDecision, which
# PostToolUse doesn't act on -- registered there, a $10,000 refund sails
# through untouched. Same two functions, swapped events, completely
# different (and broken) behavior.


## 🎓 Practice Scenario (from the module)

> An agent occasionally processes international transfers without required
> compliance checks. The compliance team requires 100% enforcement. The
> current system uses prompts, achieving ~95% success. What's the correct
> approach?
>
> - A. A `PreToolUse` hook blocking `transfer_funds` until `aml_check` returns a verified pass
> - B. Add detailed AML instructions to the system prompt, with examples and warnings
> - C. A `PostToolUse` hook flagging completed transfers that skipped AML, for manual review
> - D. Train the agent with few-shot examples of the correct AML workflow
>
> **Answer: A.** B and D are both still prompt-shaped and still probabilistic.
> C has the right idea but the wrong direction — by `PostToolUse`, the
> transfer has already gone through; flagging it after the fact doesn't undo
> a compliance violation that already happened.


## 🏆 Key Takeaways for Exam Prep

1. **Hook direction determines function** — `PostToolUse` transforms after
   execution; `PreToolUse` enforces before it. The exam tests this
   distinction relentlessly.
2. **Deterministic vs. probabilistic is still the tradeoff** — hooks cost
   engineering complexity and buy 100% compliance; prompts are cheap and
   cap out around 90–95%.
3. **Normalization removes a whole class of model error** — heterogeneous
   formats stop being the model's problem the moment a `PostToolUse` hook
   standardizes them.
4. **Prevention beats reaction** — `PreToolUse` can stop a side effect from
   happening at all; `PostToolUse` can only ever react to one that already did.
5. **The exam's favorite distractor** is a prompt-based (or wrong-direction
   hook) "fix" for a scenario that explicitly demands 100% enforcement.
   Recognize it and reject it, every time.


---

## 🎉 Quick-Fire Recap — See If It Stuck

You normalized three genuinely different data formats with one hook, and
watched a refund get denied and an AML-gated transfer get denied-then-allowed
— all through real SDK hook dispatch, not a description of how it should work.

**1. In one line: what's the difference between what `PreToolUse` and
`PostToolUse` can each actually do?**
> 💡 `PreToolUse` can prevent a tool call from happening at all.
> `PostToolUse` can only change what the model sees afterward — the tool has
> already run either way.

**2. Someone wants to "block" refunds over $500 using a `PostToolUse` hook.
What's wrong with that plan?**
> 💡 By `PostToolUse`, `process_refund` has already executed — the refund is
> already gone. There's no `permissionDecision` at that stage because there's
> nothing left to permit or deny.

**3. In your own Task 3 run, did the final summary treat all three tools'
dates/statuses consistently, or did anything slip through unnormalized?**
> 💡 Either way, check the `[PostToolUse]` lines printed from inside the hook
> itself — they show you the raw-to-normalized transformation directly, which
> is more reliable evidence than trusting the model's own summary alone.

**4. Why register `aml_check`'s `PostToolUse` hook at all, instead of just
trusting the model to remember it ran the check?**
> 💡 Because "the model remembers" is a prompt-shaped guarantee, and this is
> a compliance requirement. The hook writes the pass into real Python state
> that the `PreToolUse` gate on `transfer_funds` can check deterministically,
> regardless of what the model recalls or forgets.

**5. A teammate suggests stronger, more detailed prompt wording to hit 100%
AML compliance. What's the one-sentence reason that won't work?**
> 💡 Prompts are probabilistic by nature — better wording can push the
> success rate up, but nothing about a prompt can guarantee it, and 100% is
> what's actually required here.

**6. You accidentally register the refund-threshold hook under `PostToolUse`
instead of `PreToolUse`. What actually happens on a $10,000 refund?**
> 💡 It goes through. `PostToolUse` has no `permissionDecision` mechanism —
> the hook could log or flag the refund, but nothing stops `process_refund`
> from executing first.

---

### 🚀 Nice work.

Five modules into Domain 1 — you've now built enforcement by hand (1.4) and
with real SDK hooks (this one), on top of both a hand-rolled loop (1.1) and
the real subagent primitive (1.3). Onward to **1.6 — Task Decomposition
Strategies**.
